<a href="https://colab.research.google.com/github/NotfromEQ/Step_Up/blob/Training_Model_Playground/Try_ImageClassiffication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Handwritten Digits Prediction

ลองฝึกใช้ ImageClassification

In [52]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Setup & Data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_loader = DataLoader(datasets.MNIST('./data', train=True, download=True, transform=transform), batch_size=64, shuffle=True)

# 2. Simple CNN (สูตรมาตรฐาน)
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = SimpleCNN().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9) # เปลี่ยนมาใช้ SGD บ้างเพื่อแก้โรคเลขซ้ำ
criterion = nn.CrossEntropyLoss()

# 3. Training Loop (เทรน 2 รอบไปเลย!)
model.train()
for epoch in range(2):
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 200 == 0:
            print(f'Epoch {epoch} | Loss: {loss.item():.4f}')

print("Training Finished!")

Epoch 0 | Loss: 2.2881
Epoch 0 | Loss: 0.1751
Epoch 0 | Loss: 0.0438
Epoch 0 | Loss: 0.0690
Epoch 0 | Loss: 0.0402
Epoch 1 | Loss: 0.0087
Epoch 1 | Loss: 0.1151
Epoch 1 | Loss: 0.0457
Epoch 1 | Loss: 0.0618
Epoch 1 | Loss: 0.0370
Training Finished!


In [53]:
model.eval()
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    output = model(images)
    preds = output.argmax(dim=1)

# ดู 10 ตัวแรก
print(f"AI ทายว่า: {preds[:10].tolist()}")
print(f"ค่าจริงคือ: {labels[:10].tolist()}")

# เช็คว่าตรงกันกี่ตัวใน 64 ตัว (1 Batch)
correct = (preds == labels).sum().item()
print(f"ทายถูก {correct}/64 รูป")

AI ทายว่า: [3, 1, 5, 1, 8, 8, 8, 4, 7, 1]
ค่าจริงคือ: [3, 1, 5, 1, 8, 8, 8, 4, 7, 1]
ทายถูก 64/64 รูป


Save Model

In [54]:
torch.save(model.state_dict(), 'my_super_cnn.pth')
print("เซฟสมองระดับเทพไว้เรียบร้อย!")

เซฟสมองระดับเทพไว้เรียบร้อย!
